# Total Snippets Extraction
This notebook extract all opium mentions from the full library to prepare opium snippets for topic modelling (see 3_analysis).

In [5]:
import os
import re
import pandas as pd
import spacy
import ipywidgets as widgets
from tqdm.auto import tqdm

from IPython.display import display, HTML

# Load spaCy model
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import spacy.cli
    spacy.cli.download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')
    
import sys
sys.path.append("..")

from utils.constants import KEYWORDS

In [6]:
# Setup Paths
metadata_path = '../../data/metadata_files/GP_opium_filtered_1870_1920.parquet' # Changed this
fulltext_path = '../../../../../Downloads/history/opium_books_fulltext'

output_parquet = '../../data/snippets/total_snippets.parquet'
output_csv = '../../data/snippets/total_snippets.csv'

window_size = 100

## 1. Extracting context windows out of all books

In [3]:
book_ids = pd.read_parquet(metadata_path)["Etext Number"]
book_ids

0          16
1          24
2          27
3          36
4          44
        ...  
3738    74736
3739    74886
3740    74956
3741    75246
3742    75497
Name: Etext Number, Length: 3743, dtype: int64

In [4]:
results = []
if os.path.exists(fulltext_path):
    for book_id in book_ids:
    #os.listdir(metadata_path):
        filepath = os.path.join(fulltext_path, str(book_id))
        if os.path.isfile(filepath):
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
                text_lower = text.lower()
                for keyword in KEYWORDS:
                    pattern = r'\b' + re.escape(keyword) + r'\b'
                    for match in re.finditer(pattern, text_lower):
                        idx = match.start()
                        left_text = text[:idx]
                        right_text = text[idx + len(keyword):]
                        
                        left_words = left_text.split()
                        right_words = right_text.split()
                        
                        context_left = ' '.join(left_words[-window_size:])
                        context_right = ' '.join(right_words[:window_size])
                        
                        # Simple bounding box handling for text window
                        text_start = max(0, idx - 800)
                        text_end = min(len(text), idx + len(keyword) + 800)
                        raw_snippet = text[text_start:text_end] 
                        
                        results.append({
                            'Book_ID': book_id,
                            'Keyword': keyword,
                            'Left_Context': context_left,
                            'Right_Context': context_right,
                            'Full_Context': f"{context_left} {text[idx:idx+len(keyword)]} {context_right}"
                        })
                        
df = pd.DataFrame(results)
print(f"Found {len(df)} keyword mentions across all {len(book_ids)} books.")


KeyboardInterrupt: 

In [ ]:
# To overwrite the total snippet files, uncomment:

df[['Book_ID', 'Keyword', 'Full_Context']].to_csv(output_csv)
df[['Book_ID', 'Keyword', 'Full_Context']].to_parquet(output_parquet)

## 2. Manual Inspection Viewer
Use the slider to browse through the extracted snippets. The target keyword is highlighted.

In [ ]:
pd.set_option('display.max_colwidth', None)
def view_snippet(index):
    if len(df) == 0:
        print("No snippets found.")
        return
    row = df.iloc[index]
    html_out = f"""
    <div style='font-family: Georgia, serif; font-size: 16px; line-height: 1.6; max-width: 800px; padding: 20px; border: 1px solid #ccc; border-radius: 5px; background: #f9f9f9;'>
        <h4>Book ID: {row['Book_ID']} | Keyword: <span style='color: dimgrey;'>{row['Keyword'].upper()}</span></h4>
        <hr>
        <p>
            {row['Left_Context']} 
            <span style='background-color: #ffeb3b; font-weight: bold; padding: 0 4px;'>{row['Keyword']}</span> 
            {row['Right_Context']}
        </p>
    </div>
    """
    display(HTML(html_out))

In [ ]:
if len(df) > 0:
    slider = widgets.IntSlider(min=0, max=len(df)-1, step=1, description='Snippet:', layout=widgets.Layout(width='800px'))
    widgets.interact(view_snippet, index=slider)

interactive(children=(IntSlider(value=0, description='Snippet:', layout=Layout(width='800px'), max=3856), Outp…

## 3. Entity & POS Exploration (Optional)
Analyze the named entities (PEOPLE, ORG, LOC) and adjectives co-occurring with opium keywords across the entire dataset.

In [ ]:
all_entities = []
all_adjectives = []

for text in tqdm(df['Full_Context'], desc="Processing NLP Contexts"):
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ['PERSON', 'ORG', 'GPE', 'LOC', 'FAC', 'PRODUCT']:
            all_entities.append((ent.label_, ent.text.strip()))
    for token in doc:
        if token.pos_ == 'ADJ' and not token.is_stop and token.is_alpha:
            all_adjectives.append(token.lemma_.lower())

ent_counts = Counter(all_entities).most_common(20)
adj_counts = Counter(all_adjectives).most_common(20)

print("\n--- Top 20 Named Entities in Contexts ---")
for e, c in ent_counts:
    print(f"  {e[0]:<7}: {e[1]} ({c})")

print("\n--- Top 20 Adjectives in Contexts ---")
for a, c in adj_counts:
    print(f"  {a:<15}: {c}")


Processing NLP Contexts:   0%|          | 0/3857 [00:00<?, ?it/s]


--- Top 20 Named Entities in Contexts ---
  PERSON : Poppy (555)
  PERSON : Iglesias (157)
  GPE    : London (142)
  PERSON : Rickman (105)
  GPE    : China (92)
  PERSON : Rita (85)
  PERSON : Kennedy (83)
  PERSON : Bliss (83)
  PERSON : Chinaman (79)
  PERSON : Medjora (73)
  GPE    : New York (71)
  PERSON : Munson (70)
  PERSON : Sin (69)
  GPE    : England (68)
  PERSON : De Quincey (64)
  PERSON : Dominic Iglesias (63)
  PERSON : Grace (61)
  GPE    : Paris (59)
  PERSON : Ricky (59)
  PERSON : Jim (58)

--- Top 20 Adjectives in Contexts ---
  little         : 1376
  good           : 888
  old            : 874
  great          : 761
  black          : 596
  long           : 475
  white          : 469
  young          : 428
  new            : 368
  small          : 364
  bad            : 332
  poor           : 299
  poppy          : 288
  red            : 274
  dark           : 261
  certain        : 248
  dear           : 246
  dead           : 237
  open           : 235
  heav

## Authors extraction

In [12]:
all_snippets_path = "../../data/snippets/total_snippets.csv"
df_snippets = pd.read_csv(all_snippets_path, index_col=0).rename(columns={"Book_ID":"Etext Number"})

books_metadata_path = "../../data/metadata_files/metadata_1870_1920_with_gender.csv"
df_metadata = pd.read_csv(books_metadata_path, index_col=0)

df_snippets = df_snippets.merge(df_metadata, on="Etext Number")
df_snippets = df_snippets[~df_snippets["Keyword"].isin(["poppy", "black smoke"])]
df_snippets

,Etext Number,Keyword,Full_Context,Title,Authors,LoCC,Bookshelves,Subjects,rights,Published Year,Normalised Authors,Author Gender,Author Nationality,Author Birth,Author Death,Author Info Source,Author Info Found,Number of books by author
24,44,opium,and scared you! Nothing to cry about. I’m the ...,The Song of the Lark,"Cather, Willa",PS,Opera; Browsing: Culture/Civilization/Society;...,Opera -- Fiction ; Chicago (Ill.) -- Fiction ;...,Public domain in the USA.,1915,Willa Cather,female,American,1873.0,1947.0,Wikipedia,True,7
29,119,paregoric,"Artist 1 Latinist TRANSPORTATION, ETC. 27 Port...",A Tramp Abroad,"Twain, Mark",PS,Browsing: Culture/Civilization/Society; Browsi...,Europe -- Fiction ; Americans -- Europe -- Fic...,Public domain in the USA.,1880,Mark Twain,male,American,1835.0,1910.0,Wikipedia,True,100
30,119,paregoric,us for quite a siege--and did they suppose Zer...,A Tramp Abroad,"Twain, Mark",PS,Browsing: Culture/Civilization/Society; Browsi...,Europe -- Fiction ; Americans -- Europe -- Fic...,Public domain in the USA.,1880,Mark Twain,male,American,1835.0,1910.0,Wikipedia,True,100
31,119,paregoric,that a cow would naturally know more than a gu...,A Tramp Abroad,"Twain, Mark",PS,Browsing: Culture/Civilization/Society; Browsi...,Europe -- Fiction ; Americans -- Europe -- Fic...,Public domain in the USA.,1880,Mark Twain,male,American,1835.0,1910.0,Wikipedia,True,100
32,119,paregoric,estimate of the elevation of the several local...,A Tramp Abroad,"Twain, Mark",PS,Browsing: Culture/Civilization/Society; Browsi...,Europe -- Fiction ; Americans -- Europe -- Fic...,Public domain in the USA.,1880,Mark Twain,male,American,1835.0,1910.0,Wikipedia,True,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3762,73115,opium,over a grill that sent up occasional blasts of...,The house of bondage,"Kauffman, Reginald Wright",PS,Browsing: Culture/Civilization/Society; Browsi...,New York (N.Y.) -- Fiction ; PS ; Prostitutes ...,Public domain in the USA.,1912,Reginald Wright Kauffman,male,NaN,1877.0,1959.0,"OpenLibrary, Wikidata",True,4
3763,73115,morphine,blasts of heat from the basement. The rapidity...,The house of bondage,"Kauffman, Reginald Wright",PS,Browsing: Culture/Civilization/Society; Browsi...,New York (N.Y.) -- Fiction ; PS ; Prostitutes ...,Public domain in the USA.,1912,Reginald Wright Kauffman,male,NaN,1877.0,1959.0,"OpenLibrary, Wikidata",True,4
3765,73334,opium,but does that signify? Many English have spoke...,Routledge rides alone,"Comfort, Will Levington",PS,Browsing: History - Warfare; Browsing: Literat...,"International relations -- Fiction ; Liaoyang,...",Public domain in the USA.,1910,Will Levington Comfort,male,American,1878.0,1932.0,Wikipedia,True,9
3767,73588,opium,"It is the sound of an infant giving tongue, an...",The crow's-nest,"Duncan, Sara Jeannette",PS,Browsing: Culture/Civilization/Society; Browsi...,Invalids -- Fiction ; India -- History -- Brit...,Public domain in the USA.,1901,Sara Jeannette Duncan,female,"Canadian, Anglo-Indian",1861.0,1922.0,Wikipedia,True,7


In [13]:
summary = pd.DataFrame({
    "nb_opium_books": df_snippets.groupby("Authors")["Etext Number"].nunique(),
    "total_nb_snippets": df_snippets.groupby("Authors").size(),
})

snippets_per_book = (
    df_snippets
    .groupby(["Authors", "Etext Number"])
    .size()
    .reset_index(name="snippet_count")
)

# Add book titles
book_titles = (
    df_metadata[["Etext Number", "Title"]]
    .drop_duplicates("Etext Number")
)

snippets_per_book = snippets_per_book.merge(
    book_titles,
    on="Etext Number",
    how="left"
)

summary["snippets_per_book"] = (
    snippets_per_book
    .groupby("Authors")
    .apply(
        lambda g: {
            (row["Etext Number"], row["Title"]): row["snippet_count"]
            for _, row in g.iterrows()
        },
        include_groups=False
    )
)

summary["max_nb_snippets_per_book"] = (
    summary["snippets_per_book"].apply(lambda d: max(d.values()))
)

summary = summary.sort_values("total_nb_snippets", ascending=False)

sub_df = (
    df_metadata[["Authors", "Author Gender", "Author Nationality", "Number of books by author"]]
    .drop_duplicates()
)

summary = summary.merge(sub_df, on="Authors")

summary

,Authors,nb_opium_books,total_nb_snippets,snippets_per_book,max_nb_snippets_per_book,Author Gender,Author Nationality,Number of books by author
0,"Rohmer, Sax",7,193,"{(1182, 'Dope'): 87, (1183, 'The Return of Dr....",87,male,English,13
1,"Ottolengui, Rodrigues",1,166,"{(32985, 'A Modern Wizard'): 166}",166,male,American,2
2,"Kipling, Rudyard",14,76,"{(1858, 'Plain Tales from the Hills'): 4, (216...",10,male,"English, British",35
3,"Roe, Edward Payson",4,74,"{(5433, 'Without a Home'): 70, (6090, 'What Ca...",70,male,American,14
4,"Reeve, Arthur B. (Arthur Benjamin)",6,73,"{(5007, 'The Poisoned Pen'): 13, (5054, 'The D...",28,male,American,14
...,...,...,...,...,...,...,...,...
457,"Lloyd, Nelson",1,1,"{(23741, 'David Malcolm'): 1}",1,male,NaN,3
458,"Lockhart, Caroline",1,1,"{(23304, 'The Lady Doc'): 1}",1,female,American,4
459,"Long, John Luther",1,1,"{(33616, 'The Way of the Gods'): 1}",1,male,American,1
460,"Longfellow, Henry Wadsworth",1,1,"{(5436, 'Hyperion'): 1}",1,male,American,8


In [14]:
df_snippets[df_snippets["Authors"]=="Reed, Myrtle"]

,Etext Number,Keyword,Full_Context,Title,Authors,LoCC,Bookshelves,Subjects,rights,Published Year,Normalised Authors,Author Gender,Author Nationality,Author Birth,Author Death,Author Info Source,Author Info Found,Number of books by author
1624,12672,anodyne,"She mused, ironically, upon the permanence of ...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1625,12672,laudanum,"fashioning--was only a dream, from which she a...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1626,12672,laudanum,"set aside. Every one came to this, sooner or l...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1627,12672,laudanum,"Miss Hitty, tasted of the soup. A little later...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1628,12672,laudanum,her skirts as she had done when she came in. I...,A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1629,12672,laudanum,"crossed the tracks again, at the deserted poin...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1630,12672,laudanum,"side of the house was, as yet, untouched, and ...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1631,12672,laudanum,about among the rubbish. By a flash of intuiti...,A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1632,12672,laudanum,"shall clasp thee again, And with God be the re...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9
1633,12672,laudanum,"easier. He rapped once, with hesitation, then ...",A Spinner in the Sun,"Reed, Myrtle",PS,Browsing: Literature; Browsing: Fiction,Fiction ; PS,Public domain in the USA.,1906,Myrtle Reed,female,American,1874.0,1911.0,"Wikipedia, OpenLibrary",True,9


In [15]:
path_to_opium_authors = "../../data/metadata_files/opium_authors_snippet_nb.csv"
#summary.to_csv(path_to_opium_authors)